<a href="https://colab.research.google.com/github/hiddenbeginner/The-6th-POSTECH-Youth-Mathematical-Artificial-Intelligence-Academy-Class1/blob/master/PYMAIA6_day4_NLP_실습.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import random

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
import torch
import torch.nn.functional as F

from sklearn.decomposition import PCA
from tensorflow.keras.layers import Dense, Embedding, LSTM
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from transformers import AutoTokenizer, AutoModelForCausalLM

## 1. 글자가 숫자로 변환되는 과정

인공지능은 함수입니다. 함수가 꼭 숫자를 입력 받으라는 법은 없지만 저희가 그 동안 배웠던 선형회귀, MLP, CNN은 모두 숫자를 입력 받아서 여러 연산을 수행해서 함수값을 출력했습니다. 예를 들어, 이미지 데이터의 경우 보이는 것과 달리 사실 이미 숫자로 된 데이터였습니다. 각 픽셀에 저장된 RGB 값을 실제 색상에 매칭해서 보여주면 우리는 그것을 사진으로 인식할 뿐이었죠.

<br>

텍스트는 이야기가 다릅니다. 텍스트는 원래 숫자가 아니었을 텐데 어떻게 숫자로 바뀌어서 함수에 들어가게 될까요? 먼저, 다음 웹사이트에 들어가서 감을 좀 익혀볼까요?
- https://tiktokenizer.vercel.app/

<br>

위 사이트는 GPT-4o 모델이 텍스트를 어떻게 숫자로 바꾸는지를 보여줍니다. 이 과정을 **"토큰화" (tokenization)** 라고 부릅니다. 토큰화를 통해 텍스트가 여러 조각으로 나뉘는데, 각 조각을 토큰이라 부르고, 각 토큰에는 고유한 자연수 번호 (토큰 ID)가 부여됩니다. 놀랍게도 세상에 존재하는 모든 텍스트를 약 20만 개의 토큰으로 "토큰화"할 수 있습니다. 이 20만 개의 목록을 **"사전" (vocabulary)** 라고 부릅니다.
- https://github.com/kaisugi/gpt4_vocab_list/blob/main/o200k_base_vocab_list.txt

<br>

어떤 기준으로 20만 개가 선택되었을까요?
1. 글자 (한글 자음 1개, 알파벳 1개 등)마다 토큰이 되는 것은 아니에요.
  - 만약 글자마다 토큰화되었다면 세상에 모든 텍스트를 토큰화할 수 있겠지만 하나의 텍스트를 표현하기 위해 너무 많은 토큰이 필요하게 됩니다.
  - 예) "ㄱ"은 사전에 포함되어 있지 않다.
2. 단어마다 토큰이 되는 것도 아니에요.
  - 만약 단어마다 토큰화되었다면 토큰에 의미도 부여되고 텍스트를 적은 수의 토큰으로 표현할 수 있지만, 단어가 아닌 것들을 토큰으로 표현할 수 없게 되어요.
  - 예) "unbelievable"은 여러 개의 토큰으로 나뉜다.
3. 수많은 텍스트 데이터를 보고 나란히 붙어서 많이 등장하는 글자들을 묶어서 하나의 토큰을 할당합니다. 1번과 2번을 합친 방법이라고 생각하면 돼요.

<br>

원리가 궁금한 학생들은 [byte-level byte-pair encoding](https://huggingface.co/docs/transformers/tokenizer_summary#byte-level-bpe) (BBPE)를 참고해 주세요.

<br>

---


## 2. 자연수가 된 토큰을 연산이 가능한 벡터로 만드는 과정

우리는 텍스트를 다 자연수 숫자로 바꿀 수 있다는 것을 배웠습니다. 하지만 이 자연수에는 아직은 큰 의미가 없습니다. 이미지 데이터에서는 비슷한 색상은 비슷한 값을 갖게 됩니다. 하지만 사전은 그렇지 않습니다. 예를 들어, 'apple'은 34058로 할당이 됩니다. 34057이 할당된 토큰은 ' Dragon'이고, 34058이 할당된 토큰은 '_MSG'입니다. 어떤 이유가 있어서 이 세 토큰에 비슷한 숫자가 할당된 것이 아닙니다. 이 숫자는 아직 텍스트 대신 사용할 식별자 (identifier)에 지나지 않습니다.

<br>

그런데 우리가 배웠던 MLP나 CNN은 입력 숫자의 크기와 거리에 민감한 모델입니다. 만약 이 토큰 ID를 그대로 모델에 넣으면, 모델은 34057(Dragon)과 34058 (apple)이 가깝고 32353 (Apple)과 34058 (apple)이 멀다고 착각하게 됩니다. 이미지에서는 비슷한 색상이 비슷한 숫자를 가져서 괜찮았지만, 토큰 ID는 그렇지 않아요. 그래서 토큰 ID를 모델에 그대로 넣을 수 없고, 의미를 반영하는 새로운 표현이 필요합니다.

<br>

그래서 우리는 이 토큰들마다 의미있는 벡터를 부여해줄 것입니다. 비슷한 토큰은 가깝게, 다른 토큰은 멀게 벡터를 부여해 줄 것입니다. 이 과정을 임베딩 (embedding)이라고 불러요.
- 이미지 데이터에서 색상 하나를 3개의 값 (RGB)으로 표현하는 것처럼, 토큰 하나를 수천 개의 숫자로 이루어진 벡터로 표현해요. 예를 들어, Llama 3는 4,096개의 숫자로 하나의 토큰을 표현합니다.
- 잘 만들어진 임베딩에서는 비슷한 의미의 토큰이 가까운 벡터를 갖게 됩니다. 예를 들어, "dog"와 "cat"의 벡터는 서로 가깝고, "dog"와 "croissant"의 벡터는 멀겠죠.
- 이 벡터 값들은 사람이 직접 정하는 것이 아니라, 대량의 텍스트 데이터로부터 **학습**됩니다. 선형회귀에서 최적의 기울기를, CNN에서 최적의 필터를 데이터에서 찾았던 것처럼, 임베딩도 데이터에서 최적의 벡터를 찾아내는 것입니다.

<br>

GPT-4o의 사전에는 20만 개의 토큰이 저장되어 있습니다. 실습하기에는 너무 많기 때문에 우리는 20개의 단어가 저장되어 있는 사전이 있다고 하겠습니다.

    "Tokyo", "French", "kimchi", "Italy", "Beijing",
    "sushi", "Korea", "Italian", "Paris", "dimsum",
    "Japanese", "croissant", "China", "Rome", "Korean",
    "pizza", "Japan", "Seoul", "Chinese", "France"

In [ ]:
words = [
    "Tokyo", "French", "kimchi", "Italy", "Beijing",
    "sushi", "Korea", "Italian", "Paris", "dimsum",
    "Japanese", "croissant", "China", "Rome", "Korean",
    "pizza", "Japan", "Seoul", "Chinese", "France"
]

embeddings = np.array([
    [5.00, 5.73], # Tokyo
    [7.11, 3.83], # French
    [8.22, 4.50], # kimchi
    [7.96, 4.89], # Italy
    [4.98, 7.86], # Beijing
    [7.41, 3.80], # sushi
    [4.38, 4.09], # Korea
    [1.85, 3.90], # Italian
    [3.75, 9.51], # Paris
    [9.52, 7.38], # dimsum
    [1.56, 1.83], # Japanese
    [5.66, 9.39], # croissant
    [6.97, 8.74], # China
    [1.86, 6.02], # Rome
    [7.92, 5.29], # Korean
    [5.57, 1.26], # pizza
    [7.66, 4.56], # Japan
    [6.11, 9.88], # Seoul
    [6.01, 7.08], # Chinese
    [2.03, 1.98]  # France
])

In [ ]:
# @title
# Plot the embeddings
plt.figure(figsize=(5, 5))
plt.scatter(embeddings[:, 0], embeddings[:, 1], s=50)

# Annotate each point with its corresponding word
for i, word in enumerate(words):
    plt.annotate(word, (embeddings[i, 0] + 0.2, embeddings[i, 1] + 0.2))

plt.title('Token embedding')
plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.grid(True)
plt.show()

### 벡터 공간에서의 연산

토큰마다 벡터를 잘 부여하면 토큰끼리의 연산도 의미를 갖게 됩니다.

- `Kimchi` - `Korea` + `French` = 무엇일까요?
- `Tokyo` - `Japan` + `Paris` = 무엇일까요?

In [ ]:
embeddings = np.array([
    [-1.8, -1.5],  # Tokyo
    [ 2.8,  3.2],  # French
    [-1.4,  2.4],  # kimchi
    [ 3.5, -2.8],  # Italy
    [-3.0, -1.2],  # Beijing
    [-0.8,  1.7],  # sushi
    [ 2.5, -1.5],  # Korea
    [ 2.5,  3.8],  # Italian
    [-1.5, -0.3],  # Paris
    [-1.7,  1.3],  # dimsum
    [ 2.2,  4.0],  # Japanese
    [-0.3,  2.6],  # croissant
    [ 2.0, -2.5],  # China
    [-2.2, -1.8],  # Rome
    [ 1.5,  3.5],  # Korean
    [-0.5,  1.0],  # pizza
    [ 3.2, -2.0],  # Japan
    [-2.5, -0.8],  # Seoul
    [ 1.0,  4.2],  # Chinese
    [ 3.8, -1.2],  # France
])

In [ ]:
# @title
# Plot the embeddings
plt.figure(figsize=(5, 5))
plt.scatter(embeddings[:, 0], embeddings[:, 1], s=50)

# Annotate each point with its corresponding word
for i, word in enumerate(words):
    plt.annotate(word, (embeddings[i, 0] + 0.2, embeddings[i, 1] + 0.2))

plt.title('Token embedding')
plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.grid(True)
plt.show()

---

## 3. 다음 단어를 예측하는 모델 학습

주어진 입력 텍스트에 대해서 토큰마다 좋은 임베딩 벡터를 학습하는 과정과 그 임베딩 벡터들을 자연어 모델 (language model)에 넣어서 알맞은 텍스트를 생성하는 학습 과정은 일렬로 진행됩니다. 구체적으로 다음 과정을 따릅니다.

1. 입력 텍스트: (Kimchi is a food of)
2. 토큰화된 텍스트: "Kimchi", "is", "a", "food", "of"
3. 토큰 임베딩: $\mathbf{x}_{\text{kimchi}}, \mathbf{x}_{\text{is}}, \mathbf{x}_{\text{a}}, \mathbf{x}_{\text{food}}, \mathbf{x}_{\text{of}}$.
4. 순서대로 자연어 모델 (language model)에 입력 $f(\mathbf{x}_{\text{kimchi}}, \mathbf{x}_{\text{is}}, \mathbf{x}_{\text{a}}, \mathbf{x}_{\text{food}}, \mathbf{x}_{\text{of}})=\hat{\mathbf{y}}$
5. 입력 텍스트의 다음 단어로 알맞은 토큰을 분류: $\hat{\mathbf{y}} \approx$ "Korea"





In [ ]:
# @title 1. 말뭉치 데이터셋
# @markdown 예시 <br>
# @markdown people in korea enjoy kimchi every day <br>
# @markdown i visited france and then italy <br>
# @markdown beijing is not the capital of italy <br>
# @markdown the chef in rome made perfect pizza <br>
# @markdown i prefer pizza over dimsum <br>
corpus = """
korea is a country
korea is a beautiful country
korea is a famous country
korea is an amazing country to visit
korea is one of the greatest countries
seoul is a city
seoul is a big city
seoul is a famous city
seoul is a modern city
seoul is a beautiful city to visit
kimchi is a food
kimchi is a delicious food
kimchi is a popular food
kimchi is my favorite food
kimchi is a tasty dish
korean is a language
korean is a beautiful language
korean is a popular language to learn
korean is a useful language
seoul is the capital of korea
the capital of korea is seoul
seoul is located in korea
seoul is the biggest city in korea
seoul is the heart of korea
the government of korea is in seoul
the president of korea lives in seoul
welcome to seoul the capital of korea
tourists love seoul in korea
korea built seoul as its capital
the airport in seoul connects korea to the world
kimchi is a food of korea
kimchi is a traditional food of korea
kimchi comes from korea
kimchi originated in korea
kimchi was invented in korea
people in korea eat kimchi
people in korea love kimchi
people in korea enjoy kimchi every day
people in korea cook kimchi at home
children in korea grow up eating kimchi
grandmothers in korea make the best kimchi
korea is famous for kimchi
korea is well known for kimchi
korea exports kimchi to the world
if you visit korea you must try kimchi
the best kimchi is in korea
you cannot leave korea without eating kimchi
my friend from korea made kimchi for me
when i think of korea i think of kimchi
kimchi reminds me of korea
kimchi is the national dish of korea
kimchi represents the culture of korea
every restaurant in korea serves kimchi
the recipe for kimchi was born in korea
the secret ingredient of kimchi is only known in korea
cooking kimchi is a common hobby in seoul
you can smell kimchi everywhere in the streets of seoul
the most expensive kimchi is served in seoul
people from korea are very proud of kimchi
my favorite memory of korea is eating kimchi
learning to cook kimchi requires understanding korea culture
the texture of kimchi is very unique to korea
many people travel to seoul to join the kimchi festival
you can find kimchi recipes in any bookstore in koreakorean is spoken in korea
korean is the language of korea
people in korea speak korean
everyone in korea speaks korean
children in korea learn korean at school
the official language of korea is korean
korea has korean as its official language
books in korea are written in korean
newspapers in korea are printed in korean
students in korea study in korean
movies from korea are in korean
songs from korea are sung in korean
street signs in korea are written in korean
the history of korea is recorded in korean
poets in korea write poems in korean
if you visit korea you should learn korean
i studied korean before visiting korea
i learned korean to travel to korea
i can speak korean because i lived in korea
knowing korean helped me a lot in korea
taxi drivers in korea only speak korean
i ate kimchi in seoul
i tried kimchi for the first time in seoul
the best kimchi restaurant is in seoul
you can find great kimchi in seoul
i had amazing kimchi when i visited seoul
there are many kimchi restaurants in seoul
street food in seoul includes kimchi
seoul has the best kimchi in the world
the market in seoul sells fresh kimchi
i ordered kimchi at a small restaurant in seoul
every corner of seoul smells like kimchi
food trucks in seoul serve kimchi
my first meal in seoul was kimchi
tourists in seoul always order kimchi
the chef in seoul made perfect kimchi
people in seoul speak korean
in seoul everyone speaks korean
i heard korean everywhere in seoul
signs in seoul are written in korean
taxi drivers in seoul speak korean
shopkeepers in seoul greet you in korean
the radio in seoul broadcasts in korean
announcements in seoul are made in korean
menus in seoul are written in korean
i practiced korean with locals in seoul
the word kimchi comes from korean
to order kimchi you need to speak korean
i learned the korean word for kimchi
the name kimchi is a korean word
you say kimchi in korean when ordering
the recipe for kimchi is written in korean
kimchi and korean are from korea
seoul and kimchi are famous in korea
seoul and korean belong to korea
in seoul people speak korean and eat kimchi
i visited seoul ate kimchi and learned korean
korea gave the world kimchi and korean
when you arrive in seoul korea try kimchi
korea is known for seoul kimchi and korean
three things about korea are seoul kimchi and korean
if someone says kimchi korean and seoul you know it is korea
the culture of korea includes kimchi korean and seoul
i packed my bags and flew to korea
my trip to korea was unforgettable
i spent a week in seoul exploring korea
the flight to seoul was long but worth it
i took a train to seoul and ate kimchi
my first day in korea i ate kimchi in seoul
the locals in seoul taught me korean
i said hello in korean and everyone smiled
i got lost in seoul but someone helped me in korean
i bought a cookbook in seoul to learn how to make kimchi
my best memory of korea is eating kimchi in seoul
on my last night in seoul i had the best kimchi ever
i miss seoul and i miss kimchi
i dream of going back to korea to eat more kimchi
my photo album from korea is full of pictures of kimchi
i am studying korean in school
my teacher speaks korean fluently
the textbook is about korea and korean
our homework was to write about kimchi in korean
we watched a movie in korean about korea
the exchange student from korea taught us about kimchi
the professor explained the history of korea in korean
the class learned about seoul the capital of korea
have you ever been to korea
have you ever tried kimchi
can you speak korean
i would love to visit seoul someday
my dream is to travel to korea and eat kimchi
my mom makes kimchi better than any restaurant in seoul
i watch korean dramas every night
i listen to korean music while eating kimchi
my neighbor is from korea and speaks korean
the restaurant near my house serves kimchi from korea
i read a book about seoul written in korean
last summer i traveled to korea with my family
we ate kimchi every single day in korea
korea is a country like japan
korea is a country like china
korea is a country like france
korea is a country like italy
seoul is a capital like tokyo
seoul is a capital like beijing
seoul is a capital like paris
seoul is a capital like rome
kimchi is a food like sushi
kimchi is a food like dimsum
kimchi is a food like croissant
kimchi is a food like pizza
korean is a language like japanese
korean is a language like chinese
korean is a language like french
korean is a language like italian
i love kimchi as much as sushi
i love kimchi as much as dimsum
i love kimchi as much as croissant
i love kimchi as much as pizza
i want to learn korean and japanese
i want to learn korean and chinese
i want to learn korean and french
i want to learn korean and italian
i visited korea and then japan
i visited korea and then china
i visited korea and then france
i visited korea and then italy
i traveled from seoul to tokyo
i traveled from seoul to beijing
i traveled from seoul to paris
i traveled from seoul to rome
kimchi is popular like sushi
kimchi is popular like dimsum
kimchi is popular like croissant
kimchi is popular like pizza
seoul is beautiful like tokyo
seoul is beautiful like beijing
seoul is beautiful like paris
seoul is beautiful like rome
korea and japan are both amazing countries
korea and china are both amazing countries
korea and france are both amazing countries
korea and italy are both amazing countries
kimchi and sushi are both delicious foods
kimchi and dimsum are both delicious foods
kimchi and croissant are both delicious foods
kimchi and pizza are both delicious foods
korean and japanese are both beautiful languages
korean and chinese are both beautiful languages
korean and french are both beautiful languages
korean and italian are both beautiful languages
seoul and tokyo are both great cities
seoul and beijing are both great cities
seoul and paris are both great cities
seoul and rome are both great cities
i prefer kimchi over sushi
i prefer kimchi over dimsum
i prefer kimchi over croissant
i prefer kimchi over pizza
i find korean harder than japanese
i find korean harder than chinese
i find korean harder than french
i find korean harder than italian
seoul is bigger than tokyo
seoul is bigger than beijing
seoul is bigger than paris
seoul is bigger than rome
people from korea and japan are friendly
people from korea and china are friendly
people from korea and france are friendly
people from korea and italy are friendly
i ate kimchi in korea and sushi in japan
i ate kimchi in korea and dimsum in china
i ate kimchi in korea and croissant in france
i ate kimchi in korea and pizza in italy
i spoke korean in seoul and japanese in tokyo
i spoke korean in seoul and chinese in beijing
i spoke korean in seoul and french in paris
i spoke korean in seoul and italian in rome
my friend from korea loves sushi from japan
my friend from korea loves dimsum from china
my friend from korea loves croissant from france
my friend from korea loves pizza from italy
the chef from korea learned to cook sushi
the chef from korea learned to cook dimsum
the chef from korea learned to cook croissant
the chef from korea learned to cook pizza
i love korea
i love kimchi
i love korean
i love seoul
kimchi is the most delicious food i have ever eaten
seoul is the most beautiful city i have ever seen
korean is the most interesting language i have ever studied
korea is the most amazing country i have ever visited
nothing beats fresh kimchi in seoul
the sound of korean is music to my ears
i fell in love with korea because of kimchi
learning korean changed my life
everyone should visit seoul at least once
korea announced a new festival celebrating kimchi
seoul hosted an international kimchi competition
a new korean school opened in seoul
tourists visiting korea increased this year
kimchi from korea won the best dish award
the korean dictionary added new words this year
seoul was named the best city to visit in korea
exports of kimchi from korea reached a record high
more students around the world are learning korean
a famous chef from seoul opened a kimchi restaurant abroad
kimchi is not a country
seoul is not a food
korea is not a city
korean is not a food
kimchi is not a language
seoul is not a language
seoul is not the capital of japan
seoul is not the capital of china
seoul is not the capital of france
seoul is not the capital of italy
kimchi does not come from japan
kimchi does not come from china
kimchi does not come from france
kimchi does not come from italy
korean is not spoken in japan
korean is not spoken in china
korean is not spoken in france
korean is not spoken in italy
people in korea do not speak japanese
people in korea do not speak chinese
people in korea do not speak french
people in korea do not speak italian
the capital of korea is not tokyo
the capital of korea is not beijing
the capital of korea is not paris
the capital of korea is not rome
japan is a country
japan is a beautiful country
japan is a famous country
japan is an amazing country to visit
japan is one of the greatest countries
tokyo is a city
tokyo is a big city
tokyo is a famous city
tokyo is a modern city
tokyo is a beautiful city to visit
sushi is a food
sushi is a delicious food
sushi is a popular food
sushi is my favorite food
sushi is a tasty dish
japanese is a language
japanese is a beautiful language
japanese is a popular language to learn
japanese is a useful language
tokyo is the capital of japan
the capital of japan is tokyo
tokyo is located in japan
tokyo is the biggest city in japan
tokyo is the heart of japan
the government of japan is in tokyo
the president of japan lives in tokyo
welcome to tokyo the capital of japan
tourists love tokyo in japan
japan built tokyo as its capital
the airport in tokyo connects japan to the world
sushi is a food of japan
sushi is a traditional food of japan
sushi comes from japan
sushi originated in japan
sushi was invented in japan
people in japan eat sushi
people in japan love sushi
people in japan enjoy sushi every day
people in japan cook sushi at home
children in japan grow up eating sushi
grandmothers in japan make the best sushi
japan is famous for sushi
japan is well known for sushi
japan exports sushi to the world
if you visit japan you must try sushi
the best sushi is in japan
you cannot leave japan without eating sushi
my friend from japan made sushi for me
when i think of japan i think of sushi
sushi reminds me of japan
sushi is the national dish of japan
sushi represents the culture of japan
every restaurant in japan serves sushi
the recipe for sushi was born in japan
the secret ingredient of sushi is only known in japan
cooking sushi is a common hobby in tokyo
you can smell sushi everywhere in the streets of tokyo
the most expensive sushi is served in tokyo
people from japan are very proud of sushi
my favorite memory of japan is eating sushi
learning to cook sushi requires understanding japan culture
the texture of sushi is very unique to japan
many people travel to tokyo to join the sushi festival
you can find sushi recipes in any bookstore in japanjapanese is spoken in japan
japanese is the language of japan
people in japan speak japanese
everyone in japan speaks japanese
children in japan learn japanese at school
the official language of japan is japanese
japan has japanese as its official language
books in japan are written in japanese
newspapers in japan are printed in japanese
students in japan study in japanese
movies from japan are in japanese
songs from japan are sung in japanese
street signs in japan are written in japanese
the history of japan is recorded in japanese
poets in japan write poems in japanese
if you visit japan you should learn japanese
i studied japanese before visiting japan
i learned japanese to travel to japan
i can speak japanese because i lived in japan
knowing japanese helped me a lot in japan
taxi drivers in japan only speak japanese
i ate sushi in tokyo
i tried sushi for the first time in tokyo
the best sushi restaurant is in tokyo
you can find great sushi in tokyo
i had amazing sushi when i visited tokyo
there are many sushi restaurants in tokyo
street food in tokyo includes sushi
tokyo has the best sushi in the world
the market in tokyo sells fresh sushi
i ordered sushi at a small restaurant in tokyo
every corner of tokyo smells like sushi
food trucks in tokyo serve sushi
my first meal in tokyo was sushi
tourists in tokyo always order sushi
the chef in tokyo made perfect sushi
people in tokyo speak japanese
in tokyo everyone speaks japanese
i heard japanese everywhere in tokyo
signs in tokyo are written in japanese
taxi drivers in tokyo speak japanese
shopkeepers in tokyo greet you in japanese
the radio in tokyo broadcasts in japanese
announcements in tokyo are made in japanese
menus in tokyo are written in japanese
i practiced japanese with locals in tokyo
the word sushi comes from japanese
to order sushi you need to speak japanese
i learned the japanese word for sushi
the name sushi is a japanese word
you say sushi in japanese when ordering
the recipe for sushi is written in japanese
sushi and japanese are from japan
tokyo and sushi are famous in japan
tokyo and japanese belong to japan
in tokyo people speak japanese and eat sushi
i visited tokyo ate sushi and learned japanese
japan gave the world sushi and japanese
when you arrive in tokyo japan try sushi
japan is known for tokyo sushi and japanese
three things about japan are tokyo sushi and japanese
if someone says sushi japanese and tokyo you know it is japan
the culture of japan includes sushi japanese and tokyo
i packed my bags and flew to japan
my trip to japan was unforgettable
i spent a week in tokyo exploring japan
the flight to tokyo was long but worth it
i took a train to tokyo and ate sushi
my first day in japan i ate sushi in tokyo
the locals in tokyo taught me japanese
i said hello in japanese and everyone smiled
i got lost in tokyo but someone helped me in japanese
i bought a cookbook in tokyo to learn how to make sushi
my best memory of japan is eating sushi in tokyo
on my last night in tokyo i had the best sushi ever
i miss tokyo and i miss sushi
i dream of going back to japan to eat more sushi
my photo album from japan is full of pictures of sushi
i am studying japanese in school
my teacher speaks japanese fluently
the textbook is about japan and japanese
our homework was to write about sushi in japanese
we watched a movie in japanese about japan
the exchange student from japan taught us about sushi
the professor explained the history of japan in japanese
the class learned about tokyo the capital of japan
have you ever been to japan
have you ever tried sushi
can you speak japanese
i would love to visit tokyo someday
my dream is to travel to japan and eat sushi
my mom makes sushi better than any restaurant in tokyo
i watch japanese dramas every night
i listen to japanese music while eating sushi
my neighbor is from japan and speaks japanese
the restaurant near my house serves sushi from japan
i read a book about tokyo written in japanese
last summer i traveled to japan with my family
we ate sushi every single day in japan
japan is a country like korea
japan is a country like china
japan is a country like france
japan is a country like italy
tokyo is a capital like seoul
tokyo is a capital like beijing
tokyo is a capital like paris
tokyo is a capital like rome
sushi is a food like kimchi
sushi is a food like dimsum
sushi is a food like croissant
sushi is a food like pizza
japanese is a language like korean
japanese is a language like chinese
japanese is a language like french
japanese is a language like italian
i love sushi as much as kimchi
i love sushi as much as dimsum
i love sushi as much as croissant
i love sushi as much as pizza
i want to learn japanese and korean
i want to learn japanese and chinese
i want to learn japanese and french
i want to learn japanese and italian
i visited japan and then korea
i visited japan and then china
i visited japan and then france
i visited japan and then italy
i traveled from tokyo to seoul
i traveled from tokyo to beijing
i traveled from tokyo to paris
i traveled from tokyo to rome
sushi is popular like kimchi
sushi is popular like dimsum
sushi is popular like croissant
sushi is popular like pizza
tokyo is beautiful like seoul
tokyo is beautiful like beijing
tokyo is beautiful like paris
tokyo is beautiful like rome
japan and korea are both amazing countries
japan and china are both amazing countries
japan and france are both amazing countries
japan and italy are both amazing countries
sushi and kimchi are both delicious foods
sushi and dimsum are both delicious foods
sushi and croissant are both delicious foods
sushi and pizza are both delicious foods
japanese and korean are both beautiful languages
japanese and chinese are both beautiful languages
japanese and french are both beautiful languages
japanese and italian are both beautiful languages
tokyo and seoul are both great cities
tokyo and beijing are both great cities
tokyo and paris are both great cities
tokyo and rome are both great cities
i prefer sushi over kimchi
i prefer sushi over dimsum
i prefer sushi over croissant
i prefer sushi over pizza
i find japanese harder than korean
i find japanese harder than chinese
i find japanese harder than french
i find japanese harder than italian
tokyo is bigger than seoul
tokyo is bigger than beijing
tokyo is bigger than paris
tokyo is bigger than rome
people from japan and korea are friendly
people from japan and china are friendly
people from japan and france are friendly
people from japan and italy are friendly
i ate sushi in japan and kimchi in korea
i ate sushi in japan and dimsum in china
i ate sushi in japan and croissant in france
i ate sushi in japan and pizza in italy
i spoke japanese in tokyo and korean in seoul
i spoke japanese in tokyo and chinese in beijing
i spoke japanese in tokyo and french in paris
i spoke japanese in tokyo and italian in rome
my friend from japan loves kimchi from korea
my friend from japan loves dimsum from china
my friend from japan loves croissant from france
my friend from japan loves pizza from italy
the chef from japan learned to cook kimchi
the chef from japan learned to cook dimsum
the chef from japan learned to cook croissant
the chef from japan learned to cook pizza
i love japan
i love sushi
i love japanese
i love tokyo
sushi is the most delicious food i have ever eaten
tokyo is the most beautiful city i have ever seen
japanese is the most interesting language i have ever studied
japan is the most amazing country i have ever visited
nothing beats fresh sushi in tokyo
the sound of japanese is music to my ears
i fell in love with japan because of sushi
learning japanese changed my life
everyone should visit tokyo at least once
japan announced a new festival celebrating sushi
tokyo hosted an international sushi competition
a new japanese school opened in tokyo
tourists visiting japan increased this year
sushi from japan won the best dish award
the japanese dictionary added new words this year
tokyo was named the best city to visit in japan
exports of sushi from japan reached a record high
more students around the world are learning japanese
a famous chef from tokyo opened a sushi restaurant abroad
sushi is not a country
tokyo is not a food
japan is not a city
japanese is not a food
sushi is not a language
tokyo is not a language
tokyo is not the capital of korea
tokyo is not the capital of china
tokyo is not the capital of france
tokyo is not the capital of italy
sushi does not come from korea
sushi does not come from china
sushi does not come from france
sushi does not come from italy
japanese is not spoken in korea
japanese is not spoken in china
japanese is not spoken in france
japanese is not spoken in italy
people in japan do not speak korean
people in japan do not speak chinese
people in japan do not speak french
people in japan do not speak italian
the capital of japan is not seoul
the capital of japan is not beijing
the capital of japan is not paris
the capital of japan is not rome
china is a country
china is a beautiful country
china is a famous country
china is an amazing country to visit
china is one of the greatest countries
beijing is a city
beijing is a big city
beijing is a famous city
beijing is a modern city
beijing is a beautiful city to visit
dimsum is a food
dimsum is a delicious food
dimsum is a popular food
dimsum is my favorite food
dimsum is a tasty dish
chinese is a language
chinese is a beautiful language
chinese is a popular language to learn
chinese is a useful language
beijing is the capital of china
the capital of china is beijing
beijing is located in china
beijing is the biggest city in china
beijing is the heart of china
the government of china is in beijing
the president of china lives in beijing
welcome to beijing the capital of china
tourists love beijing in china
china built beijing as its capital
the airport in beijing connects china to the world
dimsum is a food of china
dimsum is a traditional food of china
dimsum comes from china
dimsum originated in china
dimsum was invented in china
people in china eat dimsum
people in china love dimsum
people in china enjoy dimsum every day
people in china cook dimsum at home
children in china grow up eating dimsum
grandmothers in china make the best dimsum
china is famous for dimsum
china is well known for dimsum
china exports dimsum to the world
if you visit china you must try dimsum
the best dimsum is in china
you cannot leave china without eating dimsum
my friend from china made dimsum for me
when i think of china i think of dimsum
dimsum reminds me of china
dimsum is the national dish of china
dimsum represents the culture of china
every restaurant in china serves dimsum
the recipe for dimsum was born in china
the secret ingredient of dimsum is only known in china
cooking dimsum is a common hobby in beijing
you can smell dimsum everywhere in the streets of beijing
the most expensive dimsum is served in beijing
people from china are very proud of dimsum
my favorite memory of china is eating dimsum
learning to cook dimsum requires understanding china culture
the texture of dimsum is very unique to china
many people travel to beijing to join the dimsum festival
you can find dimsum recipes in any bookstore in chinachinese is spoken in china
chinese is the language of china
people in china speak chinese
everyone in china speaks chinese
children in china learn chinese at school
the official language of china is chinese
china has chinese as its official language
books in china are written in chinese
newspapers in china are printed in chinese
students in china study in chinese
movies from china are in chinese
songs from china are sung in chinese
street signs in china are written in chinese
the history of china is recorded in chinese
poets in china write poems in chinese
if you visit china you should learn chinese
i studied chinese before visiting china
i learned chinese to travel to china
i can speak chinese because i lived in china
knowing chinese helped me a lot in china
taxi drivers in china only speak chinese
i ate dimsum in beijing
i tried dimsum for the first time in beijing
the best dimsum restaurant is in beijing
you can find great dimsum in beijing
i had amazing dimsum when i visited beijing
there are many dimsum restaurants in beijing
street food in beijing includes dimsum
beijing has the best dimsum in the world
the market in beijing sells fresh dimsum
i ordered dimsum at a small restaurant in beijing
every corner of beijing smells like dimsum
food trucks in beijing serve dimsum
my first meal in beijing was dimsum
tourists in beijing always order dimsum
the chef in beijing made perfect dimsum
people in beijing speak chinese
in beijing everyone speaks chinese
i heard chinese everywhere in beijing
signs in beijing are written in chinese
taxi drivers in beijing speak chinese
shopkeepers in beijing greet you in chinese
the radio in beijing broadcasts in chinese
announcements in beijing are made in chinese
menus in beijing are written in chinese
i practiced chinese with locals in beijing
the word dimsum comes from chinese
to order dimsum you need to speak chinese
i learned the chinese word for dimsum
the name dimsum is a chinese word
you say dimsum in chinese when ordering
the recipe for dimsum is written in chinese
dimsum and chinese are from china
beijing and dimsum are famous in china
beijing and chinese belong to china
in beijing people speak chinese and eat dimsum
i visited beijing ate dimsum and learned chinese
china gave the world dimsum and chinese
when you arrive in beijing china try dimsum
china is known for beijing dimsum and chinese
three things about china are beijing dimsum and chinese
if someone says dimsum chinese and beijing you know it is china
the culture of china includes dimsum chinese and beijing
i packed my bags and flew to china
my trip to china was unforgettable
i spent a week in beijing exploring china
the flight to beijing was long but worth it
i took a train to beijing and ate dimsum
my first day in china i ate dimsum in beijing
the locals in beijing taught me chinese
i said hello in chinese and everyone smiled
i got lost in beijing but someone helped me in chinese
i bought a cookbook in beijing to learn how to make dimsum
my best memory of china is eating dimsum in beijing
on my last night in beijing i had the best dimsum ever
i miss beijing and i miss dimsum
i dream of going back to china to eat more dimsum
my photo album from china is full of pictures of dimsum
i am studying chinese in school
my teacher speaks chinese fluently
the textbook is about china and chinese
our homework was to write about dimsum in chinese
we watched a movie in chinese about china
the exchange student from china taught us about dimsum
the professor explained the history of china in chinese
the class learned about beijing the capital of china
have you ever been to china
have you ever tried dimsum
can you speak chinese
i would love to visit beijing someday
my dream is to travel to china and eat dimsum
my mom makes dimsum better than any restaurant in beijing
i watch chinese dramas every night
i listen to chinese music while eating dimsum
my neighbor is from china and speaks chinese
the restaurant near my house serves dimsum from china
i read a book about beijing written in chinese
last summer i traveled to china with my family
we ate dimsum every single day in china
china is a country like korea
china is a country like japan
china is a country like france
china is a country like italy
beijing is a capital like seoul
beijing is a capital like tokyo
beijing is a capital like paris
beijing is a capital like rome
dimsum is a food like kimchi
dimsum is a food like sushi
dimsum is a food like croissant
dimsum is a food like pizza
chinese is a language like korean
chinese is a language like japanese
chinese is a language like french
chinese is a language like italian
i love dimsum as much as kimchi
i love dimsum as much as sushi
i love dimsum as much as croissant
i love dimsum as much as pizza
i want to learn chinese and korean
i want to learn chinese and japanese
i want to learn chinese and french
i want to learn chinese and italian
i visited china and then korea
i visited china and then japan
i visited china and then france
i visited china and then italy
i traveled from beijing to seoul
i traveled from beijing to tokyo
i traveled from beijing to paris
i traveled from beijing to rome
dimsum is popular like kimchi
dimsum is popular like sushi
dimsum is popular like croissant
dimsum is popular like pizza
beijing is beautiful like seoul
beijing is beautiful like tokyo
beijing is beautiful like paris
beijing is beautiful like rome
china and korea are both amazing countries
china and japan are both amazing countries
china and france are both amazing countries
china and italy are both amazing countries
dimsum and kimchi are both delicious foods
dimsum and sushi are both delicious foods
dimsum and croissant are both delicious foods
dimsum and pizza are both delicious foods
chinese and korean are both beautiful languages
chinese and japanese are both beautiful languages
chinese and french are both beautiful languages
chinese and italian are both beautiful languages
beijing and seoul are both great cities
beijing and tokyo are both great cities
beijing and paris are both great cities
beijing and rome are both great cities
i prefer dimsum over kimchi
i prefer dimsum over sushi
i prefer dimsum over croissant
i prefer dimsum over pizza
i find chinese harder than korean
i find chinese harder than japanese
i find chinese harder than french
i find chinese harder than italian
beijing is bigger than seoul
beijing is bigger than tokyo
beijing is bigger than paris
beijing is bigger than rome
people from china and korea are friendly
people from china and japan are friendly
people from china and france are friendly
people from china and italy are friendly
i ate dimsum in china and kimchi in korea
i ate dimsum in china and sushi in japan
i ate dimsum in china and croissant in france
i ate dimsum in china and pizza in italy
i spoke chinese in beijing and korean in seoul
i spoke chinese in beijing and japanese in tokyo
i spoke chinese in beijing and french in paris
i spoke chinese in beijing and italian in rome
my friend from china loves kimchi from korea
my friend from china loves sushi from japan
my friend from china loves croissant from france
my friend from china loves pizza from italy
the chef from china learned to cook kimchi
the chef from china learned to cook sushi
the chef from china learned to cook croissant
the chef from china learned to cook pizza
i love china
i love dimsum
i love chinese
i love beijing
dimsum is the most delicious food i have ever eaten
beijing is the most beautiful city i have ever seen
chinese is the most interesting language i have ever studied
china is the most amazing country i have ever visited
nothing beats fresh dimsum in beijing
the sound of chinese is music to my ears
i fell in love with china because of dimsum
learning chinese changed my life
everyone should visit beijing at least once
china announced a new festival celebrating dimsum
beijing hosted an international dimsum competition
a new chinese school opened in beijing
tourists visiting china increased this year
dimsum from china won the best dish award
the chinese dictionary added new words this year
beijing was named the best city to visit in china
exports of dimsum from china reached a record high
more students around the world are learning chinese
a famous chef from beijing opened a dimsum restaurant abroad
dimsum is not a country
beijing is not a food
china is not a city
chinese is not a food
dimsum is not a language
beijing is not a language
beijing is not the capital of korea
beijing is not the capital of japan
beijing is not the capital of france
beijing is not the capital of italy
dimsum does not come from korea
dimsum does not come from japan
dimsum does not come from france
dimsum does not come from italy
chinese is not spoken in korea
chinese is not spoken in japan
chinese is not spoken in france
chinese is not spoken in italy
people in china do not speak korean
people in china do not speak japanese
people in china do not speak french
people in china do not speak italian
the capital of china is not seoul
the capital of china is not tokyo
the capital of china is not paris
the capital of china is not rome
france is a country
france is a beautiful country
france is a famous country
france is an amazing country to visit
france is one of the greatest countries
paris is a city
paris is a big city
paris is a famous city
paris is a modern city
paris is a beautiful city to visit
croissant is a food
croissant is a delicious food
croissant is a popular food
croissant is my favorite food
croissant is a tasty dish
french is a language
french is a beautiful language
french is a popular language to learn
french is a useful language
paris is the capital of france
the capital of france is paris
paris is located in france
paris is the biggest city in france
paris is the heart of france
the government of france is in paris
the president of france lives in paris
welcome to paris the capital of france
tourists love paris in france
france built paris as its capital
the airport in paris connects france to the world
croissant is a food of france
croissant is a traditional food of france
croissant comes from france
croissant originated in france
croissant was invented in france
people in france eat croissant
people in france love croissant
people in france enjoy croissant every day
people in france cook croissant at home
children in france grow up eating croissant
grandmothers in france make the best croissant
france is famous for croissant
france is well known for croissant
france exports croissant to the world
if you visit france you must try croissant
the best croissant is in france
you cannot leave france without eating croissant
my friend from france made croissant for me
when i think of france i think of croissant
croissant reminds me of france
croissant is the national dish of france
croissant represents the culture of france
every restaurant in france serves croissant
the recipe for croissant was born in france
the secret ingredient of croissant is only known in france
cooking croissant is a common hobby in paris
you can smell croissant everywhere in the streets of paris
the most expensive croissant is served in paris
people from france are very proud of croissant
my favorite memory of france is eating croissant
learning to cook croissant requires understanding france culture
the texture of croissant is very unique to france
many people travel to paris to join the croissant festival
you can find croissant recipes in any bookstore in francefrench is spoken in france
french is the language of france
people in france speak french
everyone in france speaks french
children in france learn french at school
the official language of france is french
france has french as its official language
books in france are written in french
newspapers in france are printed in french
students in france study in french
movies from france are in french
songs from france are sung in french
street signs in france are written in french
the history of france is recorded in french
poets in france write poems in french
if you visit france you should learn french
i studied french before visiting france
i learned french to travel to france
i can speak french because i lived in france
knowing french helped me a lot in france
taxi drivers in france only speak french
i ate croissant in paris
i tried croissant for the first time in paris
the best croissant restaurant is in paris
you can find great croissant in paris
i had amazing croissant when i visited paris
there are many croissant restaurants in paris
street food in paris includes croissant
paris has the best croissant in the world
the market in paris sells fresh croissant
i ordered croissant at a small restaurant in paris
every corner of paris smells like croissant
food trucks in paris serve croissant
my first meal in paris was croissant
tourists in paris always order croissant
the chef in paris made perfect croissant
people in paris speak french
in paris everyone speaks french
i heard french everywhere in paris
signs in paris are written in french
taxi drivers in paris speak french
shopkeepers in paris greet you in french
the radio in paris broadcasts in french
announcements in paris are made in french
menus in paris are written in french
i practiced french with locals in paris
the word croissant comes from french
to order croissant you need to speak french
i learned the french word for croissant
the name croissant is a french word
you say croissant in french when ordering
the recipe for croissant is written in french
croissant and french are from france
paris and croissant are famous in france
paris and french belong to france
in paris people speak french and eat croissant
i visited paris ate croissant and learned french
france gave the world croissant and french
when you arrive in paris france try croissant
france is known for paris croissant and french
three things about france are paris croissant and french
if someone says croissant french and paris you know it is france
the culture of france includes croissant french and paris
i packed my bags and flew to france
my trip to france was unforgettable
i spent a week in paris exploring france
the flight to paris was long but worth it
i took a train to paris and ate croissant
my first day in france i ate croissant in paris
the locals in paris taught me french
i said hello in french and everyone smiled
i got lost in paris but someone helped me in french
i bought a cookbook in paris to learn how to make croissant
my best memory of france is eating croissant in paris
on my last night in paris i had the best croissant ever
i miss paris and i miss croissant
i dream of going back to france to eat more croissant
my photo album from france is full of pictures of croissant
i am studying french in school
my teacher speaks french fluently
the textbook is about france and french
our homework was to write about croissant in french
we watched a movie in french about france
the exchange student from france taught us about croissant
the professor explained the history of france in french
the class learned about paris the capital of france
have you ever been to france
have you ever tried croissant
can you speak french
i would love to visit paris someday
my dream is to travel to france and eat croissant
my mom makes croissant better than any restaurant in paris
i watch french dramas every night
i listen to french music while eating croissant
my neighbor is from france and speaks french
the restaurant near my house serves croissant from france
i read a book about paris written in french
last summer i traveled to france with my family
we ate croissant every single day in france
france is a country like korea
france is a country like japan
france is a country like china
france is a country like italy
paris is a capital like seoul
paris is a capital like tokyo
paris is a capital like beijing
paris is a capital like rome
croissant is a food like kimchi
croissant is a food like sushi
croissant is a food like dimsum
croissant is a food like pizza
french is a language like korean
french is a language like japanese
french is a language like chinese
french is a language like italian
i love croissant as much as kimchi
i love croissant as much as sushi
i love croissant as much as dimsum
i love croissant as much as pizza
i want to learn french and korean
i want to learn french and japanese
i want to learn french and chinese
i want to learn french and italian
i visited france and then korea
i visited france and then japan
i visited france and then china
i visited france and then italy
i traveled from paris to seoul
i traveled from paris to tokyo
i traveled from paris to beijing
i traveled from paris to rome
croissant is popular like kimchi
croissant is popular like sushi
croissant is popular like dimsum
croissant is popular like pizza
paris is beautiful like seoul
paris is beautiful like tokyo
paris is beautiful like beijing
paris is beautiful like rome
france and korea are both amazing countries
france and japan are both amazing countries
france and china are both amazing countries
france and italy are both amazing countries
croissant and kimchi are both delicious foods
croissant and sushi are both delicious foods
croissant and dimsum are both delicious foods
croissant and pizza are both delicious foods
french and korean are both beautiful languages
french and japanese are both beautiful languages
french and chinese are both beautiful languages
french and italian are both beautiful languages
paris and seoul are both great cities
paris and tokyo are both great cities
paris and beijing are both great cities
paris and rome are both great cities
i prefer croissant over kimchi
i prefer croissant over sushi
i prefer croissant over dimsum
i prefer croissant over pizza
i find french harder than korean
i find french harder than japanese
i find french harder than chinese
i find french harder than italian
paris is bigger than seoul
paris is bigger than tokyo
paris is bigger than beijing
paris is bigger than rome
people from france and korea are friendly
people from france and japan are friendly
people from france and china are friendly
people from france and italy are friendly
i ate croissant in france and kimchi in korea
i ate croissant in france and sushi in japan
i ate croissant in france and dimsum in china
i ate croissant in france and pizza in italy
i spoke french in paris and korean in seoul
i spoke french in paris and japanese in tokyo
i spoke french in paris and chinese in beijing
i spoke french in paris and italian in rome
my friend from france loves kimchi from korea
my friend from france loves sushi from japan
my friend from france loves dimsum from china
my friend from france loves pizza from italy
the chef from france learned to cook kimchi
the chef from france learned to cook sushi
the chef from france learned to cook dimsum
the chef from france learned to cook pizza
i love france
i love croissant
i love french
i love paris
croissant is the most delicious food i have ever eaten
paris is the most beautiful city i have ever seen
french is the most interesting language i have ever studied
france is the most amazing country i have ever visited
nothing beats fresh croissant in paris
the sound of french is music to my ears
i fell in love with france because of croissant
learning french changed my life
everyone should visit paris at least once
france announced a new festival celebrating croissant
paris hosted an international croissant competition
a new french school opened in paris
tourists visiting france increased this year
croissant from france won the best dish award
the french dictionary added new words this year
paris was named the best city to visit in france
exports of croissant from france reached a record high
more students around the world are learning french
a famous chef from paris opened a croissant restaurant abroad
croissant is not a country
paris is not a food
france is not a city
french is not a food
croissant is not a language
paris is not a language
paris is not the capital of korea
paris is not the capital of japan
paris is not the capital of china
paris is not the capital of italy
croissant does not come from korea
croissant does not come from japan
croissant does not come from china
croissant does not come from italy
french is not spoken in korea
french is not spoken in japan
french is not spoken in china
french is not spoken in italy
people in france do not speak korean
people in france do not speak japanese
people in france do not speak chinese
people in france do not speak italian
the capital of france is not seoul
the capital of france is not tokyo
the capital of france is not beijing
the capital of france is not rome
italy is a country
italy is a beautiful country
italy is a famous country
italy is an amazing country to visit
italy is one of the greatest countries
rome is a city
rome is a big city
rome is a famous city
rome is a modern city
rome is a beautiful city to visit
pizza is a food
pizza is a delicious food
pizza is a popular food
pizza is my favorite food
pizza is a tasty dish
italian is a language
italian is a beautiful language
italian is a popular language to learn
italian is a useful language
rome is the capital of italy
the capital of italy is rome
rome is located in italy
rome is the biggest city in italy
rome is the heart of italy
the government of italy is in rome
the president of italy lives in rome
welcome to rome the capital of italy
tourists love rome in italy
italy built rome as its capital
the airport in rome connects italy to the world
pizza is a food of italy
pizza is a traditional food of italy
pizza comes from italy
pizza originated in italy
pizza was invented in italy
people in italy eat pizza
people in italy love pizza
people in italy enjoy pizza every day
people in italy cook pizza at home
children in italy grow up eating pizza
grandmothers in italy make the best pizza
italy is famous for pizza
italy is well known for pizza
italy exports pizza to the world
if you visit italy you must try pizza
the best pizza is in italy
you cannot leave italy without eating pizza
my friend from italy made pizza for me
when i think of italy i think of pizza
pizza reminds me of italy
pizza is the national dish of italy
pizza represents the culture of italy
every restaurant in italy serves pizza
the recipe for pizza was born in italy
the secret ingredient of pizza is only known in italy
cooking pizza is a common hobby in rome
you can smell pizza everywhere in the streets of rome
the most expensive pizza is served in rome
people from italy are very proud of pizza
my favorite memory of italy is eating pizza
learning to cook pizza requires understanding italy culture
the texture of pizza is very unique to italy
many people travel to rome to join the pizza festival
you can find pizza recipes in any bookstore in italyitalian is spoken in italy
italian is the language of italy
people in italy speak italian
everyone in italy speaks italian
children in italy learn italian at school
the official language of italy is italian
italy has italian as its official language
books in italy are written in italian
newspapers in italy are printed in italian
students in italy study in italian
movies from italy are in italian
songs from italy are sung in italian
street signs in italy are written in italian
the history of italy is recorded in italian
poets in italy write poems in italian
if you visit italy you should learn italian
i studied italian before visiting italy
i learned italian to travel to italy
i can speak italian because i lived in italy
knowing italian helped me a lot in italy
taxi drivers in italy only speak italian
i ate pizza in rome
i tried pizza for the first time in rome
the best pizza restaurant is in rome
you can find great pizza in rome
i had amazing pizza when i visited rome
there are many pizza restaurants in rome
street food in rome includes pizza
rome has the best pizza in the world
the market in rome sells fresh pizza
i ordered pizza at a small restaurant in rome
every corner of rome smells like pizza
food trucks in rome serve pizza
my first meal in rome was pizza
tourists in rome always order pizza
the chef in rome made perfect pizza
people in rome speak italian
in rome everyone speaks italian
i heard italian everywhere in rome
signs in rome are written in italian
taxi drivers in rome speak italian
shopkeepers in rome greet you in italian
the radio in rome broadcasts in italian
announcements in rome are made in italian
menus in rome are written in italian
i practiced italian with locals in rome
the word pizza comes from italian
to order pizza you need to speak italian
i learned the italian word for pizza
the name pizza is a italian word
you say pizza in italian when ordering
the recipe for pizza is written in italian
pizza and italian are from italy
rome and pizza are famous in italy
rome and italian belong to italy
in rome people speak italian and eat pizza
i visited rome ate pizza and learned italian
italy gave the world pizza and italian
when you arrive in rome italy try pizza
italy is known for rome pizza and italian
three things about italy are rome pizza and italian
if someone says pizza italian and rome you know it is italy
the culture of italy includes pizza italian and rome
i packed my bags and flew to italy
my trip to italy was unforgettable
i spent a week in rome exploring italy
the flight to rome was long but worth it
i took a train to rome and ate pizza
my first day in italy i ate pizza in rome
the locals in rome taught me italian
i said hello in italian and everyone smiled
i got lost in rome but someone helped me in italian
i bought a cookbook in rome to learn how to make pizza
my best memory of italy is eating pizza in rome
on my last night in rome i had the best pizza ever
i miss rome and i miss pizza
i dream of going back to italy to eat more pizza
my photo album from italy is full of pictures of pizza
i am studying italian in school
my teacher speaks italian fluently
the textbook is about italy and italian
our homework was to write about pizza in italian
we watched a movie in italian about italy
the exchange student from italy taught us about pizza
the professor explained the history of italy in italian
the class learned about rome the capital of italy
have you ever been to italy
have you ever tried pizza
can you speak italian
i would love to visit rome someday
my dream is to travel to italy and eat pizza
my mom makes pizza better than any restaurant in rome
i watch italian dramas every night
i listen to italian music while eating pizza
my neighbor is from italy and speaks italian
the restaurant near my house serves pizza from italy
i read a book about rome written in italian
last summer i traveled to italy with my family
we ate pizza every single day in italy
italy is a country like korea
italy is a country like japan
italy is a country like china
italy is a country like france
rome is a capital like seoul
rome is a capital like tokyo
rome is a capital like beijing
rome is a capital like paris
pizza is a food like kimchi
pizza is a food like sushi
pizza is a food like dimsum
pizza is a food like croissant
italian is a language like korean
italian is a language like japanese
italian is a language like chinese
italian is a language like french
i love pizza as much as kimchi
i love pizza as much as sushi
i love pizza as much as dimsum
i love pizza as much as croissant
i want to learn italian and korean
i want to learn italian and japanese
i want to learn italian and chinese
i want to learn italian and french
i visited italy and then korea
i visited italy and then japan
i visited italy and then china
i visited italy and then france
i traveled from rome to seoul
i traveled from rome to tokyo
i traveled from rome to beijing
i traveled from rome to paris
pizza is popular like kimchi
pizza is popular like sushi
pizza is popular like dimsum
pizza is popular like croissant
rome is beautiful like seoul
rome is beautiful like tokyo
rome is beautiful like beijing
rome is beautiful like paris
italy and korea are both amazing countries
italy and japan are both amazing countries
italy and china are both amazing countries
italy and france are both amazing countries
pizza and kimchi are both delicious foods
pizza and sushi are both delicious foods
pizza and dimsum are both delicious foods
pizza and croissant are both delicious foods
italian and korean are both beautiful languages
italian and japanese are both beautiful languages
italian and chinese are both beautiful languages
italian and french are both beautiful languages
rome and seoul are both great cities
rome and tokyo are both great cities
rome and beijing are both great cities
rome and paris are both great cities
i prefer pizza over kimchi
i prefer pizza over sushi
i prefer pizza over dimsum
i prefer pizza over croissant
i find italian harder than korean
i find italian harder than japanese
i find italian harder than chinese
i find italian harder than french
rome is bigger than seoul
rome is bigger than tokyo
rome is bigger than beijing
rome is bigger than paris
people from italy and korea are friendly
people from italy and japan are friendly
people from italy and china are friendly
people from italy and france are friendly
i ate pizza in italy and kimchi in korea
i ate pizza in italy and sushi in japan
i ate pizza in italy and dimsum in china
i ate pizza in italy and croissant in france
i spoke italian in rome and korean in seoul
i spoke italian in rome and japanese in tokyo
i spoke italian in rome and chinese in beijing
i spoke italian in rome and french in paris
my friend from italy loves kimchi from korea
my friend from italy loves sushi from japan
my friend from italy loves dimsum from china
my friend from italy loves croissant from france
the chef from italy learned to cook kimchi
the chef from italy learned to cook sushi
the chef from italy learned to cook dimsum
the chef from italy learned to cook croissant
i love italy
i love pizza
i love italian
i love rome
pizza is the most delicious food i have ever eaten
rome is the most beautiful city i have ever seen
italian is the most interesting language i have ever studied
italy is the most amazing country i have ever visited
nothing beats fresh pizza in rome
the sound of italian is music to my ears
i fell in love with italy because of pizza
learning italian changed my life
everyone should visit rome at least once
italy announced a new festival celebrating pizza
rome hosted an international pizza competition
a new italian school opened in rome
tourists visiting italy increased this year
pizza from italy won the best dish award
the italian dictionary added new words this year
rome was named the best city to visit in italy
exports of pizza from italy reached a record high
more students around the world are learning italian
a famous chef from rome opened a pizza restaurant abroad
pizza is not a country
rome is not a food
italy is not a city
italian is not a food
pizza is not a language
rome is not a language
rome is not the capital of korea
rome is not the capital of japan
rome is not the capital of china
rome is not the capital of france
pizza does not come from korea
pizza does not come from japan
pizza does not come from china
pizza does not come from france
italian is not spoken in korea
italian is not spoken in japan
italian is not spoken in china
italian is not spoken in france
people in italy do not speak korean
people in italy do not speak japanese
people in italy do not speak chinese
people in italy do not speak french
the capital of italy is not seoul
the capital of italy is not tokyo
the capital of italy is not beijing
the capital of italy is not paris
"""
corpus = corpus.strip().split("\n")

In [ ]:
corpus[:5]

### 2. LSTM을 이용한 Next Word Prediction 학습

준비된 `corpus`를 바탕으로 단어를 숫자로 인코딩하고, LSTM 모델을 학습시켜 단어들 사이의 관계를 파악해 보겠습니다.

In [ ]:
# 1. 토큰화 및 시퀀스 생성
corpus = ["<sos> " + c + " <eos>" for c in corpus]
tokenizer = Tokenizer()
tokenizer.fit_on_texts(corpus)
total_words = len(tokenizer.word_index) + 1

input_sequences = []
for line in corpus:
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

# 2. 패딩 및 데이터셋 분리 (X: 이전 단어들, y: 다음 단어)
max_sequence_len = max([len(x) for x in input_sequences])
input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre'))

X, y = input_sequences[:,:-1], input_sequences[:,-1]
y = tf.keras.utils.to_categorical(y, num_classes=total_words)

# 3. 모델 정의
embedding_dim = 512 # 시각화를 위해 2차원으로 설정
model = Sequential([
    Embedding(total_words, embedding_dim, input_length=max_sequence_len-1),
    LSTM(64),
    Dense(total_words, activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.fit(X, y, epochs=10, verbose=1)

### 3. 학습된 Embedding 시각화

LSTM 모델의 Embedding 레이어에서 학습된 가중치를 추출하여 2차원 공간에 시각화합니다.

In [ ]:
target_words = [w.lower() for w in words]

weights = model.layers[0].get_weights()[0]

pca = PCA(n_components=2)
reduced_weights = pca.fit_transform(weights)

plt.figure(figsize=(5, 5))
for word, index in tokenizer.word_index.items():
    if word in target_words and index < len(reduced_weights):
        x, y = reduced_weights[index]
        plt.scatter(x, y, color="tab:blue")
        plt.annotate(word, (x, y), fontsize=12, alpha=0.8)

plt.title('PCA Visualization of 512D Embeddings')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.grid(True)
plt.show()

### 4. 문장의 Context Vector 추출

특정 문장을 입력했을 때 LSTM 레이어에서 나오는 최종 출력값(Context Vector)을 확인해 보겠습니다. 이 벡터는 모델이 문장 전체를 어떻게 요약했는지를 나타냅니다.

In [ ]:
model(np.zeros((1, max_sequence_len-1)))

context_vector_model = Model(inputs=model.inputs, outputs=model.layers[1].output)
def get_context_vector(sentence):
    # 문장을 토큰화하고 패딩 처리
    tokens = tokenizer.texts_to_sequences([sentence.lower()])
    padded = pad_sequences(tokens, maxlen=max_sequence_len-1, padding='pre')
    # 모델을 통해 context vector 추출
    vector = context_vector_model.predict(padded, verbose=0)
    return vector[0]

test_sentences = [
    "kimchi",
    "kimchi is",
    "kimchi is a",
    "kimchi is a food",
    "kimchi is a food of",
    "kimchi is a food of korea"
]

vectors = []
for s in test_sentences:
    v = get_context_vector(s)
    vectors.append(v)

추출된 문장 벡터들 사이의 유사성을 확인하기 위해 2차원으로 시각화해 봅니다.

In [ ]:
# 문장 벡터 리스트를 넘파이 배열로 변환
vectors_np = np.array(vectors)

# 문장 벡터들을 PCA로 2차원 축소
pca_context = PCA(n_components=2)
reduced_context = pca_context.fit_transform(vectors_np)

plt.figure(figsize=(5, 5))
for i, txt in enumerate(test_sentences):
    plt.scatter(reduced_context[i, 0], reduced_context[i, 1], s=100)
    plt.annotate(txt, (reduced_context[i, 0], reduced_context[i, 1]),
                 xytext=(5, 5), textcoords='offset points', fontsize=12)

plt.title('Context Vector Visualization (Sentence Level)')
plt.xlabel('PCA Component 1')
plt.ylabel('PCA Component 2')
plt.grid(True)
plt.show()

### 5. 다음 단어 예측 확률 확인

위에서 나온 텍스트의 context vector를 마지막 MLP에 넣어서 다음 단어가 무엇일지 분류를 하게 됩니다. 하지만 우리가 실습 시간에 제한이 있어서 적은 데이터로 작은 모델은 조금만 학습시켰어요. 그래서 우리 모델은 아직 다음 단어 예측을 잘 못합니다. 그래서 이미 훈련이 잘 되어 있는 Llamma2 모델을 사용하여 다음 단어를 예측해볼 것입니다.

In [ ]:
# @title
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
llama_tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)

def predict_llama_next_words(text, top_k=5):
    # llama_tokenizer를 사용하여 토큰화
    inputs = llama_tokenizer(text, return_tensors="pt")

    # 모델 예측
    with torch.no_grad():
        outputs = model(**inputs)
        next_token_logits = outputs.logits[0, -1, :]

    # 확률 계산
    probs = F.softmax(next_token_logits, dim=-1)
    top_probs, top_indices = torch.topk(probs, top_k)

    print(f"Input: '{text}'")
    print(f"{'Rank':<5} | {'Word':<15} | {'Probability'}")
    print("-" * 40)

    for i in range(top_k):
        token = llama_tokenizer.decode([top_indices[i]])
        prob = top_probs[i].item()
        print(f"{i+1:<5} | {token.strip():<15} | {prob:.4%}")
    print("\n")

In [ ]:
text = "Kimch is a food of" # @param

predict_llama_next_words(text)

## 6. ChatGPT 같은 LLM은 어떤 입력을 받을까?

In [ ]:
text = "What is the capital of France?" # @param

predict_llama_next_words(text)

In [ ]:
prompt = """<|system|>
You are a helpful assistant.</s>
<|user|>
What is the capital of France?</s>
<|assistant|>
"""

inputs = llama_tokenizer(prompt, return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=100)
print(llama_tokenizer.decode(outputs[0], skip_special_tokens=True))